# Taller 3 Consumo de APIs + Power BI - Juan Pablo Arevalo

## Importacion de librerias

In [249]:
import requests
import pandas as pd
import pycountry

## Extracción de datos desde API CityBikes
 
### Base de informacion: https://api.citybik.es/v2/

In [250]:
base_url = "http://api.citybik.es/v2/networks"


response = requests.get(base_url)
if response.status_code == 200:
  print("Conexión exitosa a la API. Codigo de estado:", response.status_code)
else:
  print("Conexión fallida a la API. Codigo de estado", response.status_code)


Conexión exitosa a la API. Codigo de estado: 200


## Conversion JSON - DataFrame

In [251]:
data = response.json()
print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['networks'])


In [252]:
df = pd.DataFrame(data["networks"])

## Analisis exploratorio de los datos

In [253]:
df.head()


,id,name,location,href,company,gbfs_href,system,source,ebikes,license,scooters,instances
0,abu-dhabi-careem-bike,Abu Dhabi Careem BIKE,"{'latitude': 24.4866, 'longitude': 54.3728, 'c...",/v2/networks/abu-dhabi-careem-bike,[Careem],https://dubai.publicbikesystem.net/customer/gb...,NaN,NaN,NaN,NaN,NaN,NaN
1,acces-velo-saguenay,Accès Vélo,"{'latitude': 48.433333, 'longitude': -71.08333...",/v2/networks/acces-velo-saguenay,[PBSC Urban Solutions],https://saguenay.publicbikesystem.net/customer...,NaN,NaN,NaN,NaN,NaN,NaN
2,aksu,Aksu,"{'latitude': 41.1664, 'longitude': 80.2617, 'c...",/v2/networks/aksu,[阿克苏公共服务],NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,alba,Alba,"{'latitude': 44.716667, 'longitude': 8.083333,...",/v2/networks/alba,[Comunicare S.r.l.],NaN,Bicincittà,https://www.bicincitta.com/frmLeStazioni.aspx?...,NaN,NaN,NaN,NaN
4,albabici,AlbaBici,"{'latitude': 38.9943, 'longitude': -1.8602, 'c...",/v2/networks/albabici,[Instituto Tecnológico de Castilla y León (ITCL)],NaN,bicicard,NaN,NaN,NaN,NaN,NaN


In [254]:
print(df.shape)
df.info()

(800, 12)
<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         800 non-null    str   
 1   name       800 non-null    str   
 2   location   800 non-null    object
 3   href       800 non-null    str   
 4   company    800 non-null    object
 5   gbfs_href  272 non-null    str   
 6   system     549 non-null    str   
 7   source     212 non-null    str   
 8   ebikes     46 non-null     object
 9   license    77 non-null     object
 10  scooters   2 non-null      object
 11  instances  1 non-null      object
dtypes: object(6), str(6)
memory usage: 75.1+ KB


## Transformacion de columnas

Se realiza extraccion de la columna location, para obtener los datos: city, country, latitude, longitude

In [255]:
print(df.loc[0,"location"])

{'latitude': 24.4866, 'longitude': 54.3728, 'city': 'Abu Dhabi', 'country': 'AE'}


In [256]:
df["country"] = df["location"].apply(lambda x: x["country"])
df["city"] = df["location"].apply(lambda x: x["city"])
df["latitude"] = df["location"].apply(lambda x: x["latitude"])
df["longitude"] = df["location"].apply(lambda x: x["longitude"])

Se completa el nombre correspondiente a cada pais

In [257]:
def country_name(code):
    country = pycountry.countries.get(alpha_2=code)
    return country.name if country else code

df["country"] = df["country"].apply(country_name)

## Limpieza de datos


In [258]:
df["company"].map(type).value_counts()


company
<class 'list'>    800
Name: count, dtype: int64

In [259]:
df["company"] = df["company"].apply(lambda x: x[0] 
    if isinstance(x, list) 
    and len(x) > 0 
    else "No especificado")

Revisar filas faltantes en las columnas

-system

-ebikes

In [260]:
df["system"].map(type).value_counts()

system
<class 'str'>      549
<class 'float'>    251
Name: count, dtype: int64

In [261]:
df["system"] = df["system"].fillna("No especificado")

In [262]:
df["ebikes"].map(type).value_counts()

ebikes
<class 'float'>    754
<class 'bool'>      46
Name: count, dtype: int64

In [263]:
df["ebikes"] = df["ebikes"].fillna("No especificado")
df["ebikes"] = df["ebikes"].replace({True: "Sí"})

### Eliminar columnas innecesarias y reordenar el dataset

In [264]:
df = df.drop(columns=["id","location","href","gbfs_href","source","license","scooters","instances"])

In [265]:
df = df[["name","country","city", "company","system", "ebikes", "latitude", "longitude"]]

## Validacion final

In [266]:
df.head()

,name,country,city,company,system,ebikes,latitude,longitude
0,Abu Dhabi Careem BIKE,United Arab Emirates,Abu Dhabi,Careem,No especificado,No especificado,24.486600,54.372800
1,Accès Vélo,Canada,Saguenay,PBSC Urban Solutions,No especificado,No especificado,48.433333,-71.083333
2,Aksu,China,阿克苏市 (Aksu City),阿克苏公共服务,No especificado,No especificado,41.166400,80.261700
3,Alba,Italy,Alba,Comunicare S.r.l.,Bicincittà,No especificado,44.716667,8.083333
4,AlbaBici,Spain,Albacete,Instituto Tecnológico de Castilla y León (ITCL),bicicard,No especificado,38.994300,-1.860200


In [267]:
print(df.shape)
df.info()

(800, 8)
<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   name       800 non-null    str    
 1   country    800 non-null    str    
 2   city       800 non-null    str    
 3   company    800 non-null    str    
 4   system     800 non-null    str    
 5   ebikes     800 non-null    object 
 6   latitude   800 non-null    float64
 7   longitude  800 non-null    float64
dtypes: float64(2), object(1), str(5)
memory usage: 50.1+ KB


In [268]:
df.isna().sum()

name         0
country      0
city         0
company      0
system       0
ebikes       0
latitude     0
longitude    0
dtype: int64

In [269]:
df.nunique()

name         591
country       52
city         765
company      151
system        39
ebikes         2
latitude     793
longitude    796
dtype: int64

## Exportacion de datos CSV

In [270]:
df.to_csv("citybikes_limpio.csv",index=False,sep=";",decimal=",",encoding="utf-8-sig")